# Los primeros modelos de GrantMatch

La preparación y la división se importan de [`pipeline_proyecto.py`](./pipeline_proyecto.py), no se copian. Es clasificación: el par (award, publicación) es un financiamiento real o no. Métrica: **AUC** (área bajo la curva ROC), que no depende de un umbral; también se reporta la exactitud. Al final, la tabla queda en `resultados_proyecto.csv`.

Antes de hacer commit: **Restart** y **Run All**, para que la tabla salga de una corrida completa.

In [1]:
import time

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.pipeline import Pipeline
from pipeline_proyecto import anio_de_corte, cargar_y_dividir, construir_preparacion, RUTA_DATOS

X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir()
print("Ano de corte:", anio_de_corte(pd.read_csv(RUTA_DATOS)))
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)

Ano de corte: 2020
Entrenamiento: (11884, 9) Prueba: (4556, 9)


In [2]:
resultados = {}
entrenados = {}


def evaluar(nombre, modelo):
    """Entrena el modelo despues de la preparacion y mide la AUC en
    entrenamiento y en prueba. Si vuelves a correr la celda de un modelo,
    su fila se reemplaza."""
    pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", modelo)])
    inicio = time.perf_counter()
    pipeline.fit(X_ent, y_ent)
    segundos = time.perf_counter() - inicio
    auc_ent = roc_auc_score(y_ent, pipeline.predict_proba(X_ent)[:, 1])
    auc_prueba = roc_auc_score(y_prueba, pipeline.predict_proba(X_prueba)[:, 1])
    resultados[nombre] = {
        "modelo": nombre,
        "auc_entrenamiento": round(auc_ent, 4),
        "auc_prueba": round(auc_prueba, 4),
        "brecha": round(auc_ent - auc_prueba, 4),
        "exactitud_prueba": round(accuracy_score(y_prueba, pipeline.predict(X_prueba)), 4),
        "segundos": round(segundos, 3),
    }
    entrenados[nombre] = pipeline
    print(f"{nombre:30s} AUC entrenamiento {auc_ent:.4f}   prueba {auc_prueba:.4f}   {segundos:6.2f} s")

## Línea base

`DummyClassifier(strategy="prior")`: predice siempre la proporción de pares reales del entrenamiento. Con clases balanceadas su AUC es 0.5, lo que saca cualquier modelo que no distinga un par real de uno al azar.

In [3]:
evaluar("linea base", DummyClassifier(strategy="prior"))

linea base                     AUC entrenamiento 0.5000   prueba 0.5000     0.02 s


## Modelo lineal: regresión logística, con la regularización por defecto y una más fuerte

In [4]:
evaluar("logistica C=1", LogisticRegression(C=1.0, max_iter=1000))

logistica C=1                  AUC entrenamiento 0.8710   prueba 0.8810     0.16 s


In [5]:
evaluar("logistica C=0.01", LogisticRegression(C=0.01, max_iter=1000))

logistica C=0.01               AUC entrenamiento 0.8673   prueba 0.8791     0.03 s


## Ensamblajes: un bosque sin límite, el mismo con profundidad limitada, y gradient boosting

In [6]:
evaluar("bosque sin limite", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))

bosque sin limite              AUC entrenamiento 0.9995   prueba 0.8833     1.11 s


In [7]:
evaluar("bosque max_depth=8", RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1))

bosque max_depth=8             AUC entrenamiento 0.9081   prueba 0.8946     0.55 s


In [8]:
evaluar("gradient boosting", GradientBoostingClassifier(random_state=42))

gradient boosting              AUC entrenamiento 0.9016   prueba 0.9018     1.05 s


## Tabla de resultados

In [9]:
tabla = pd.DataFrame(resultados.values())
tabla.to_csv("resultados_proyecto.csv", index=False)
mejor = tabla.loc[tabla["auc_prueba"].idxmax()]
print(f"Mejor en prueba: {mejor['modelo']} (AUC {mejor['auc_prueba']:.4f}, "
      f"{mejor['auc_prueba'] - resultados['linea base']['auc_prueba']:+.4f} sobre la linea base)")
tabla

Mejor en prueba: gradient boosting (AUC 0.9018, +0.4018 sobre la linea base)


,modelo,auc_entrenamiento,auc_prueba,brecha,exactitud_prueba,segundos
0,linea base,0.5000,0.5000,0.0000,0.5000,0.024
1,logistica C=1,0.8710,0.8810,-0.0100,0.8349,0.158
2,logistica C=0.01,0.8673,0.8791,-0.0118,0.8325,0.034
3,bosque sin limite,0.9995,0.8833,0.1161,0.8358,1.109
4,bosque max_depth=8,0.9081,0.8946,0.0134,0.8413,0.547
5,gradient boosting,0.9016,0.9018,-0.0002,0.8420,1.054
